# NPPE-1: Multilingual Sentiment Classification
**Model:** `google/gemma-3-1b-it` fine-tuned with QLoRA 
**Task:** Binary sentiment classification (Positive=1 / Negative=0) across 13 Indian languages 
**Metric:** Macro F1-Score

## Step 1 — Install Required Packages

In [1]:
import subprocess, sys

# I install each package individually to avoid dependency conflicts
def pip_install(package):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", package])
    print(f"✓ {package}")

# I use numpy 2.0.0 because Kaggle's environment requires it
pip_install("numpy==2.0.0")

# I use transformers 4.50.0 because it's the first version with Gemma-3 support
pip_install("transformers==4.50.0")

# I use peft for LoRA adapter training
pip_install("peft==0.12.0")

# I use trl for the SFTTrainer which handles supervised fine-tuning
pip_install("trl==0.11.4")

# I use bitsandbytes for 4-bit quantization to fit the model in GPU memory
pip_install("bitsandbytes==0.45.3")

# I use accelerate for distributed training utilities
pip_install("accelerate==0.34.2")

# Supporting libraries for data handling and evaluation
pip_install("datasets")
pip_install("sentencepiece")
pip_install("protobuf")
pip_install("scikit-learn")

print("\n✅ All packages installed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.0/19.0 MB 6.6 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


✓ numpy==2.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 109.1 MB/s eta 0:00:00
✓ transformers==4.50.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 13.7 MB/s eta 0:00:00
✓ peft==0.12.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 16.1 MB/s eta 0:00:00
✓ trl==0.11.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 12.2 MB/s eta 0:00:00
✓ bitsandbytes==0.45.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 19.8 MB/s eta 0:00:00
✓ accelerate==0.34.2
✓ datasets
✓ sentencepiece
✓ protobuf
✓ scikit-learn

✅ All packages installed successfully!


## Step 2 — Imports & Configuration

In [2]:
import os, re, random, warnings, unicodedata
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
from collections import Counter

# I import the core HuggingFace libraries for model loading and training
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed,
)

# I import PEFT for LoRA — this lets me train only a small fraction of parameters
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType,
)

# I use SFTTrainer from TRL for clean supervised fine-tuning
from trl import SFTTrainer, SFTConfig
from datasets import Dataset as HFDataset
from sklearn.metrics import f1_score, classification_report

# I set a fixed seed everywhere so results are reproducible
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

# I point to the Kaggle local model path to avoid downloading from internet
MODEL_ID   = '/kaggle/input/models/google/gemma-3/transformers/gemma-3-1b-it/1'

# I use the exact competition folder path discovered from the input file list
COMP_DIR   = '/kaggle/input/competitions/nppe-dlp-2026-term-1'
OUTPUT_DIR = '/kaggle/working/gemma-sentiment'

# I increase MAX_SEQ_LEN to 512 to avoid truncating longer sentences
MAX_SEQ_LEN = 512

# I define the exact column names as specified in the dataset description
ID_COL    = 'ID'
TEXT_COL  = 'sentence'
LANG_COL  = 'language'
LABEL_COL = 'label'

# I map language codes to full names for use in prompts
LANG_MAP = {
    'as': 'Assamese', 'bd': 'Bodo',     'bn': 'Bengali',
    'gu': 'Gujarati',  'hi': 'Hindi',    'kn': 'Kannada',
    'ml': 'Malayalam', 'mr': 'Marathi',  'or': 'Odia',
    'pa': 'Punjabi',   'ta': 'Tamil',    'te': 'Telugu',
    'ur': 'Urdu'
}

# I check which device is available — we need CUDA (GPU) for this task
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'numpy version : {np.__version__}')
print(f'torch version : {torch.__version__}')
print(f'Device        : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU           : {torch.cuda.get_device_name(0)}')
    print(f'Memory        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

2026-03-07 18:29:28.056778: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772908168.231183      25 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772908168.281270      25 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772908168.709296      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772908168.709336      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772908168.709339      25 computation_placer.cc:177] computation placer alr

numpy version : 2.0.2
torch version : 2.9.0+cu126
Device        : cuda
GPU           : Tesla P100-PCIE-16GB
Memory        : 17.1 GB


## Step 3 — Load Dataset

In [3]:
# I load the train and test CSVs directly from the competition input folder
train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
test_df  = pd.read_csv(f'{COMP_DIR}/test.csv')

print(f'Train shape : {train_df.shape}')   # should be (900, 4)
print(f'Test shape  : {test_df.shape}')    # should be (100, 3)
print(f'\nTrain columns : {train_df.columns.tolist()}')
print(f'Test  columns : {test_df.columns.tolist()}')

# I check the label distribution to understand class balance
print(f'\nLabel distribution:')
print(train_df[LABEL_COL].value_counts())

# I check language distribution to understand multilingual coverage
print(f'\nLanguage distribution:')
print(train_df[LANG_COL].value_counts())

Train shape : (900, 4)
Test shape  : (100, 3)

Train columns : ['ID', 'sentence', 'label', 'language']
Test  columns : ['ID', 'sentence', 'language']

Label distribution:
label
Positive    456
Negative    444
Name: count, dtype: int64

Language distribution:
language
ta    76
hi    74
or    72
pa    72
as    71
bd    71
gu    69
ml    68
mr    67
kn    66
ur    66
bn    65
te    63
Name: count, dtype: int64


## Step 4 — Text Cleaning

In [4]:
def clean_text(text: str) -> str:
    # I apply NFC normalization to standardize Unicode characters across all scripts
    if not isinstance(text, str):
        return ''
    text = unicodedata.normalize('NFC', text)

    # I remove HTML tags which sometimes appear in web-scraped data
    text = re.sub(r'<[^>]+>', ' ', text)

    # I remove URLs and email addresses as they don't carry sentiment
    text = re.sub(r'http\S+|www\.\S+|\S+@\S+', ' ', text)

    # I normalize multiple spaces into a single space
    text = re.sub(r'[ \t]+', ' ', text).strip()

    return text

# I apply cleaning to both train and test sets
train_df['clean_text'] = train_df[TEXT_COL].apply(clean_text)
test_df['clean_text']  = test_df[TEXT_COL].apply(clean_text)

print('Text cleaning complete ✓')
print('\nSample cleaned texts:')
for _, row in train_df.head(3).iterrows():
    print(f"  [{LANG_MAP.get(row[LANG_COL]):10s} | {row[LABEL_COL]:8s}]: {row['clean_text'][:80]}")

Text cleaning complete ✓

Sample cleaned texts:
  [Punjabi    | Negative]: ਇਹ ਫਿਲਮ ਇੱਕ ਬੇਹਤਰੀਨ ਕਹਾਣੀ ਸੁਣਾਉਣ ਦਾ ਸਭ ਤੋਂ ਵਧੀਆ ਉਦਾਹਰਣ ਹੈ, ਪਰ ਇਹ ਸਭ ਨਕਾਰਾਤਮਕ ਅਰਥ
  [Punjabi    | Negative]: ਇੱਕ ਸਮਗਰੀ ਦੇ ਰੂਪ ਵਿੱਚ, ਕੋਟਿੰਗ ਤੋਂ ਬਿਨਾਂ, ਸਿਰਫ ਆਮ ਸ਼ੀਸ਼ੇ ਦੀ ਵਰਤੋਂ ਕੀਤੀ ਜਾਂਦੀ ਹੈ।
  [Malayalam  | Negative]: ബ്രിസിലുകൾ കട്ടിയുള്ള പ്ലാസ്റ്റിക് ആണ്, അതിനാൽ അവ രോമത്തിൽ തുളച്ചുകയറുന്നില്ല.


## Step 5 — Prompt Engineering

In [5]:
def build_training_prompt(sentence: str, language_code: str, label: str) -> str:
    # I include the language name in the prompt for better cross-lingual understanding
    lang_name = LANG_MAP.get(language_code, language_code)

    # I add two few-shot examples (one per class) to guide the model's output format
    # Using Hindi and Bengali examples which are common in the training data
    user_msg = (
        f"You are an expert sentiment classifier for Indian languages. "
        f"Classify the sentiment of the {lang_name} text as 'Positive' or 'Negative'.\n\n"
        f"Examples:\n"
        f"- 'बहुत अच्छा अनुभव था' → Positive\n"
        f"- 'খুব খারাপ সেবা ছিল' → Negative\n\n"
        f"Text: {sentence}\n\nSentiment:"
    )
    # I use Gemma-3's exact chat format with start_of_turn tokens
    return (
        f"<bos><start_of_turn>user\n{user_msg}<end_of_turn>\n"
        f"<start_of_turn>model\n{label}<end_of_turn>"
    )


def build_inference_prompt(sentence: str, language_code: str) -> str:
    # I use the same prompt format as training but without the label at the end
    lang_name = LANG_MAP.get(language_code, language_code)
    user_msg = (
        f"You are an expert sentiment classifier for Indian languages. "
        f"Classify the sentiment of the {lang_name} text as 'Positive' or 'Negative'.\n\n"
        f"Examples:\n"
        f"- 'बहुत अच्छा अनुभव था' → Positive\n"
        f"- 'খুব খারাপ সেবা ছিল' → Negative\n\n"
        f"Text: {sentence}\n\nSentiment:"
    )
    # I leave the model turn open so the model generates the label
    return (
        f"<bos><start_of_turn>user\n{user_msg}<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )


print('Prompt functions defined ✓')
print('\nSample training prompt:')
print(build_training_prompt(train_df['clean_text'].iloc[0][:80],
                            train_df[LANG_COL].iloc[0],
                            train_df[LABEL_COL].iloc[0]))

Prompt functions defined ✓

Sample training prompt:
<bos><start_of_turn>user
You are an expert sentiment classifier for Indian languages. Classify the sentiment of the Punjabi text as 'Positive' or 'Negative'.

Examples:
- 'बहुत अच्छा अनुभव था' → Positive
- 'খুব খারাপ সেবা ছিল' → Negative

Text: ਇਹ ਫਿਲਮ ਇੱਕ ਬੇਹਤਰੀਨ ਕਹਾਣੀ ਸੁਣਾਉਣ ਦਾ ਸਭ ਤੋਂ ਵਧੀਆ ਉਦਾਹਰਣ ਹੈ, ਪਰ ਇਹ ਸਭ ਨਕਾਰਾਤਮਕ ਅਰਥ

Sentiment:<end_of_turn>
<start_of_turn>model
Negative<end_of_turn>


## Step 6 — Load Tokenizer

In [6]:
# # I load the tokenizer from HuggingFace Hub because the local Kaggle copy
# # has a corrupted tokenizer.json file that causes a JSON parse error
# tokenizer = AutoTokenizer.from_pretrained(
#     'google/gemma-3-1b-it',
#     trust_remote_code=True,
#     use_fast=False,       # I use the slow tokenizer to avoid the JSON parse error
#     padding_side='right', # I use right padding during training (standard for causal LM)
# )

# # I set pad_token to eos_token because Gemma has no dedicated pad token
# tokenizer.pad_token    = tokenizer.eos_token
# tokenizer.pad_token_id = tokenizer.eos_token_id

# print(f'Tokenizer loaded ✓')
# print(f'Vocab size   : {tokenizer.vocab_size:,}')
# print(f'Pad token    : "{tokenizer.pad_token}" (id={tokenizer.pad_token_id})')
# print(f'Padding side : {tokenizer.padding_side}')

# # I check token lengths on a sample to confirm MAX_SEQ_LEN=512 is sufficient
# sample_lengths = [
#     len(tokenizer(
#         build_training_prompt(row['clean_text'], row[LANG_COL], row[LABEL_COL]),
#         truncation=False
#     )['input_ids'])
#     for _, row in train_df.sample(50, random_state=SEED).iterrows()
# ]
# print(f'\nToken length stats (50 samples):')
# print(f'  Mean : {np.mean(sample_lengths):.0f}')
# print(f'  Max  : {np.max(sample_lengths)}')
# print(f'  95th : {np.percentile(sample_lengths, 95):.0f}')
# print(f'  MAX_SEQ_LEN = {MAX_SEQ_LEN}')

In [7]:
# I authenticate with HuggingFace using my token stored in Kaggle secrets
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

# I log in to HuggingFace so gated models like Gemma can be accessed
from huggingface_hub import login
login(token=hf_token)
print('HuggingFace login ✓')

# I now load the tokenizer from HuggingFace Hub with authentication
tokenizer = AutoTokenizer.from_pretrained(
    'google/gemma-3-1b-it',
    token=hf_token,
    trust_remote_code=True,
    use_fast=False,
    padding_side='right',
)

tokenizer.pad_token    = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

print(f'Tokenizer loaded ✓')
print(f'Vocab size   : {tokenizer.vocab_size:,}')
print(f'Pad token    : "{tokenizer.pad_token}"')
print(f'Padding side : {tokenizer.padding_side}')

# I check token lengths to confirm MAX_SEQ_LEN=512 is sufficient
sample_lengths = [
    len(tokenizer(
        build_training_prompt(row['clean_text'], row[LANG_COL], row[LABEL_COL]),
        truncation=False
    )['input_ids'])
    for _, row in train_df.sample(50, random_state=SEED).iterrows()
]
print(f'\nToken length stats (50 samples):')
print(f'  Mean : {np.mean(sample_lengths):.0f}')
print(f'  Max  : {np.max(sample_lengths)}')
print(f'  95th : {np.percentile(sample_lengths, 95):.0f}')
print(f'  MAX_SEQ_LEN = {MAX_SEQ_LEN}')

HuggingFace login ✓


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

Tokenizer loaded ✓
Vocab size   : 262,144
Pad token    : "<eos>"
Padding side : right

Token length stats (50 samples):
  Mean : 113
  Max  : 217
  95th : 175
  MAX_SEQ_LEN = 512


## Step 7 — Load Model with 4-bit Quantization

In [8]:
# I configure 4-bit NF4 quantization to reduce the model from ~2.5GB to ~1GB
# This is essential to fit Gemma-3-1B on P100's 16GB memory alongside training
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',              # NormalFloat4 is best for LLMs
    bnb_4bit_compute_dtype=torch.bfloat16,  # I use bfloat16 for compute
    bnb_4bit_use_double_quant=True,         # Nested quantization saves extra memory
)

# I load the model from the local Kaggle path (no internet download needed)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',           # Automatically places layers on GPU
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    attn_implementation='eager', # Required for Gemma-3 compatibility
)

# I prepare the model for k-bit training by freezing base weights
# and enabling gradient computation only for LoRA adapter weights
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True  # Saves memory by recomputing activations
)

# I disable cache because it's incompatible with gradient checkpointing
model.config.use_cache = False

print(f'Model loaded ✓')
print(f'Total parameters : {sum(p.numel() for p in model.parameters()):,}')

Model loaded ✓
Total parameters : 651,005,056


## Step 8 — Apply LoRA Adapters

In [9]:
# I auto-detect all linear projection layers to apply LoRA to
# This includes attention layers (q,k,v,o) and MLP layers (gate,up,down)
linear_names = set()
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        layer = name.split('.')[-1]
        if any(k in layer for k in
               ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                'gate_proj', 'up_proj', 'down_proj']):
            linear_names.add(layer)

target_modules = sorted(linear_names) or ['q_proj', 'v_proj']
print('LoRA target modules:', target_modules)

# I configure LoRA with rank=16 which gives a good balance of
# expressiveness vs memory. Alpha=32 means effective scaling = alpha/r = 2
lora_config = LoraConfig(
    r=16,                        # Rank controls adapter capacity
    lora_alpha=32,               # Scaling factor
    target_modules=target_modules,
    lora_dropout=0.05,           # Dropout for regularization
    bias='none',                 # I don't train bias terms
    task_type=TaskType.CAUSAL_LM,
)

# I wrap the quantized model with LoRA adapters
model = get_peft_model(model, lora_config)

# I print trainable parameters to confirm only LoRA weights are being trained
model.print_trainable_parameters()

LoRA target modules: ['down_proj', 'gate_proj', 'k_proj', 'o_proj', 'q_proj', 'up_proj', 'v_proj']
trainable params: 13,045,760 || all params: 1,012,931,712 || trainable%: 1.2879


## Step 9 — Prepare Training Dataset with Padding & Masking

In [10]:
def tokenize_and_mask(examples):
    # I tokenize each prompt and apply loss masking so the model only
    # learns to predict the label token, not the instruction tokens
    full_prompts = examples['text']

    # I tokenize with padding to MAX_SEQ_LEN for consistent batch sizes
    tokenized = tokenizer(
        full_prompts,
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding='max_length',  # I pad all sequences to the same length
        return_tensors=None,
    )

    labels = []
    # I get the token ids for the model turn marker to find where label starts
    model_turn_tokens = tokenizer.encode(
        '<start_of_turn>model\n', add_special_tokens=False
    )

    for i, input_ids in enumerate(tokenized['input_ids']):
        label_ids  = list(input_ids)
        seq_len    = len(input_ids)
        mask_until = 0

        # I find the last occurrence of the model turn marker
        for pos in range(seq_len - len(model_turn_tokens)):
            if input_ids[pos:pos+len(model_turn_tokens)] == model_turn_tokens:
                mask_until = pos + len(model_turn_tokens)

        # I set label=-100 for prompt tokens so they don't contribute to loss
        # I set label=-100 for padding tokens as well
        for j in range(seq_len):
            if j < mask_until or input_ids[j] == tokenizer.pad_token_id:
                label_ids[j] = -100

        labels.append(label_ids)

    tokenized['labels'] = labels
    return tokenized


# I build formatted prompts for ALL 900 training samples
# Using full dataset (no validation split) gives the model more examples to learn from
train_df['formatted'] = train_df.apply(
    lambda r: build_training_prompt(
        r['clean_text'], r[LANG_COL], r[LABEL_COL]
    ), axis=1
)

# I convert to HuggingFace dataset format required by SFTTrainer
hf_raw = HFDataset.from_pandas(
    train_df[['formatted']].rename(columns={'formatted': 'text'})
)

# I apply tokenization and masking across all samples
hf_train = hf_raw.map(
    tokenize_and_mask,
    batched=True,
    batch_size=32,
    remove_columns=['text'],
)
hf_train.set_format('torch')

print(f'Training dataset ready ✓')
print(f'Total samples   : {len(hf_train)}')

# I verify the masking is working correctly
sample      = hf_train[0]
n_masked    = (torch.tensor(sample['labels']) == -100).sum().item()
n_supervised = (torch.tensor(sample['labels']) != -100).sum().item()
print(f'Sample 0 — masked: {n_masked} tokens, supervised: {n_supervised} tokens')

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Training dataset ready ✓
Total samples   : 900
Sample 0 — masked: 510 tokens, supervised: 2 tokens


## Step 10 — Fine-Tune with SFTTrainer

In [11]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,

    # I train for 8 epochs (increased from 4) to give the model more
    # learning iterations on the 900 samples — this improves generalization
    num_train_epochs=8,

    # I use batch size 4 with gradient accumulation of 4
    # making effective batch size = 16 which is stable for training
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,

    # I enable gradient checkpointing to save GPU memory during backprop
    gradient_checkpointing=True,

    # I use adamw_torch_fused which is stable and works well with PEFT models
    optim='adamw_torch_fused',

    # I lower the learning rate to 1e-4 (from 2e-4) for more careful learning
    # This helps the model generalize better to unseen test data
    learning_rate=1e-4,

    # I use cosine schedule for smooth learning rate decay
    lr_scheduler_type='cosine',

    # I use 10% warmup to gradually increase LR at the start of training
    warmup_ratio=0.1,
    weight_decay=0.01,

    # I use bfloat16 mixed precision for faster and memory-efficient training
    fp16=False,
    bf16=True,

    max_seq_length=MAX_SEQ_LEN,

    # I skip SFTTrainer's default dataset preparation since I did my own
    dataset_kwargs={'skip_prepare_dataset': True},

    logging_steps=25,
    save_strategy='epoch',
    save_total_limit=1,
    report_to='none',
    seed=SEED,
    dataloader_pin_memory=False,

    # I explicitly set empty label_names to avoid the PeftModel warning
    label_names=[],
)

trainer = SFTTrainer(
    model=model,
    train_dataset=hf_train,
    tokenizer=tokenizer,
    args=sft_config,
)

print('Starting fine-tuning on all 900 samples (~70 min on P100)...')
trainer.train()
print('\nFine-tuning complete ✓')

# I save the LoRA adapter weights for potential reuse
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved → {OUTPUT_DIR}')

Starting fine-tuning on all 900 samples (~70 min on P100)...


Step,Training Loss
25,4.786300
50,2.408400
75,1.868300
100,1.683700
125,1.539900
150,1.425000
175,1.413800
200,1.233500
225,1.275300
250,1.102600



Fine-tuning complete ✓
Model saved → /kaggle/working/gemma-sentiment


## Step 11 — Inference Setup

In [12]:
# I unify all model layer dtypes to bfloat16 before inference
# This fixes the 'expected scalar type Half but found BFloat16' runtime error
for param in model.parameters():
    if param.data.dtype == torch.float16:
        param.data = param.data.to(torch.bfloat16)
for buf in model.buffers():
    if buf.dtype == torch.float16:
        buf.data = buf.data.to(torch.bfloat16)

model = model.to(torch.bfloat16)
model.eval()
print('Model dtype unified ✓')
print('Dtypes:', set(p.dtype for p in model.parameters()))


def extract_label(generated_text: str) -> str:
    # I clean up the generated text and extract the sentiment label
    gen = re.sub(r'[^\w\s]', '', generated_text).strip()

    # I check for exact word matches first (most reliable)
    if re.search(r'\bpositive\b', gen, re.IGNORECASE):
        return 'Positive'
    if re.search(r'\bnegative\b', gen, re.IGNORECASE):
        return 'Negative'

    # I fall back to first character if no exact match found
    first = gen[:1].upper()
    if first == 'P': return 'Positive'
    if first == 'N': return 'Negative'

    # I default to Positive as it's slightly more common in the dataset
    return 'Positive'


def batch_predict(sentences, languages, model, tokenizer,
                  batch_size=8, max_new_tokens=8):
    model.eval()
    predictions = []
    n = len(sentences)

    for start in range(0, n, batch_size):
        end     = min(start + batch_size, n)
        batch_s = sentences[start:end]
        batch_l = languages[start:end]

        # I build inference prompts for each sentence in the batch
        prompts = [
            build_inference_prompt(s, l)
            for s, l in zip(batch_s, batch_l)
        ]

        # I switch to left padding for inference so the generated tokens
        # are always at the end of the sequence
        tokenizer.padding_side = 'left'
        inputs = tokenizer(
            prompts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LEN,
        ).to(model.device)
        tokenizer.padding_side = 'right'  # I reset back to right for training

        # I use autocast with bfloat16 to handle mixed dtype layers consistently
        with torch.no_grad():
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                outputs = model.generate(
                    input_ids=inputs['input_ids'],
                    attention_mask=inputs['attention_mask'],  # Proper masking
                    max_new_tokens=max_new_tokens,
                    do_sample=False,  # I use greedy decoding for deterministic output
                    pad_token_id=tokenizer.eos_token_id,
                )

        # I decode only the newly generated tokens (not the input prompt)
        for j in range(len(batch_s)):
            input_len = inputs['input_ids'][j].shape[0]
            decoded   = tokenizer.decode(
                outputs[j][input_len:], skip_special_tokens=True
            )
            predictions.append(extract_label(decoded))

        if (start // batch_size) % 3 == 0:
            print(f'  [{end}/{n}]  raw: "{decoded.strip()[:20]}" → {predictions[-1]}')

    return predictions

print('\nInference utilities ready ✓')

Model dtype unified ✓
Dtypes: {torch.uint8, torch.bfloat16}

Inference utilities ready ✓


## Step 12 — Run Inference on Test Set

In [13]:
print(f'Running inference on {len(test_df)} test samples...')

# I run predictions on all 100 test samples using batch processing
test_preds = batch_predict(
    test_df['clean_text'].tolist(),
    test_df[LANG_COL].tolist(),
    model, tokenizer,
    batch_size=8
)

print(f'\nTotal predictions : {len(test_preds)}')
print('Distribution      :', Counter(test_preds))

Running inference on 100 test samples...
  [8/100]  raw: "Negative<end_of_turn" → Negative
  [32/100]  raw: "Positive<end_of_turn" → Positive
  [56/100]  raw: "Negative<end_of_turn" → Negative
  [80/100]  raw: "Negative<end_of_turn" → Negative
  [100/100]  raw: "Positive<end_of_turn" → Positive

Total predictions : 100
Distribution      : Counter({'Positive': 52, 'Negative': 48})


## Step 13 — Save Submission File

In [14]:
# I map text labels to numeric format as required by the competition
# Positive → 1, Negative → 0
label_map = {'Positive': 1, 'Negative': 0}

submission = pd.DataFrame({
    'ID'   : test_df[ID_COL].values,
    'label': [label_map[p] for p in test_preds]
})

# I run sanity checks to make sure the submission is correctly formatted
assert len(submission) == len(test_df),              'Row count mismatch!'
assert list(submission.columns) == ['ID', 'label'],  'Column name mismatch!'
assert submission['label'].isin([0, 1]).all(),        'Invalid label values found!'

print('Submission preview:')
print(submission.head(10).to_string(index=False))
print(f'\nShape  : {submission.shape}')
print('Labels :', submission['label'].value_counts().to_dict())

# I save the final submission CSV to the Kaggle working directory
submission.to_csv('/kaggle/working/submission.csv', index=False)
print('\n✅ Saved → /kaggle/working/submission.csv')
print('✅ Go to right panel → Submit to Competition → Click Submit!')

Submission preview:
 ID  label
550      1
397      1
757      1
407      0
294      1
672      1
598      0
839      0
921      0
554      0

Shape  : (100, 2)
Labels : {1: 52, 0: 48}

✅ Saved → /kaggle/working/submission.csv
✅ Go to right panel → Submit to Competition → Click Submit!
